In [ ]:
import pandas as pd 
import exifread
import uuid
import requests_cache
import openmeteo_requests
import os

from pathlib import Path 
from tqdm.auto import tqdm 
from retry_requests import retry
from datetime import datetime

In [ ]:
# Source dirctory for original images / photos 
images_dir = Path("images/original")

# All .jpg and .jpeg file paths as strings
photo_paths = [str(file_path) for file_path in images_dir.glob("*") if file_path.suffix.lower() in [".jpg", ".jpeg"]]

def dms_to_decimal(dms, ref):
    degrees = float(dms.values[0].num) / float(dms.values[0].den)
    minutes = float(dms.values[1].num) / float(dms.values[1].den)
    seconds = float(dms.values[2].num) / float(dms.values[2].den)
    decimal = round(degrees + minutes / 60 + seconds / 3600, 2)
    if ref in ("S", "W"):
        decimal = -decimal
    return decimal

def format_dt(exif_dt):
    return datetime.strptime(str(exif_dt), "%Y:%m:%d %H:%M:%S")

def generate_photo_id()-> str:
    return uuid.uuid4().hex[:12]

def extract_lat_lon_time(photo_path):
    with open(photo_path, "rb") as f:
        tags = exifread.process_file(f, details=False)

    if (
        "GPS GPSLatitude" in tags
        and "GPS GPSLongitude" in tags
        and "EXIF DateTimeOriginal" in tags
    ):
        lat = dms_to_decimal(tags["GPS GPSLatitude"], str(tags["GPS GPSLatitudeRef"]))
        lon = dms_to_decimal(tags["GPS GPSLongitude"], str(tags["GPS GPSLongitudeRef"]))
        dt = format_dt(tags["EXIF DateTimeOriginal"])
        photo_id = generate_photo_id()
    else:
        lat, lon = 90.0, 0.0
        dt = datetime(2020, 1, 1, 23, 0, 0)
        photo_id = generate_photo_id()

    return {
        "id": photo_id,
        "photo_path": photo_path,
        "datetime_taken": dt,
        "lat": lat,
        "lon": lon,
    }

rows = [extract_lat_lon_time(path) for path in photo_paths]

photo_data_df = pd.DataFrame(rows, columns=["id", "photo_path", "datetime_taken", "lat", "lon"])

photo_data_df


In [ ]:
photo_data_df["hour"] = photo_data_df['datetime_taken'].dt.round('h').dt.hour

photo_data_df.head()

In [ ]:
# add weather columns before adding weather data - fill with None

weather_cols = ["temp_f", "cloud_cover"]
for col in weather_cols:
    if col not in photo_data_df.columns:
        photo_data_df[col] = None
        
print(f"Loaded {len(photo_data_df):,} records")

In [ ]:
# add weather data to dataframe 

# Setup client 
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)


def get_weather(lat, lon, date_taken, hour=0):
    lat = round(float(lat), 2)
    lon = round(float(lon), 2)
    date_taken = pd.to_datetime(date_taken).strftime("%Y-%m-%d")
    hour = int(hour) if pd.notna(hour) else 0
    hour = hour if 0 <= hour <= 23 else 0

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_taken,
        "end_date": date_taken,
        "hourly": ["temperature_2m", "cloud_cover"],
        "temperature_unit": "fahrenheit",
        "timezone": "auto",
    }

    try:
        responses = openmeteo.weather_api(
            "https://archive-api.open-meteo.com/v1/archive", params=params
        )
        hourly = responses[0].Hourly()
        temp = hourly.Variables(0).ValuesAsNumpy()
        cloud = hourly.Variables(1).ValuesAsNumpy()

        return {
            "temp_f": round(float(temp[hour]), 1) if len(temp) > hour else None,
            "cloud_cover": round(float(cloud[hour]), 1) if len(cloud) > hour else None,
        }
    except Exception as e:
        print(f"Weather API error ({lat}, {lon}, {date_taken} {hour}): {e}")
        return {"temp_f": None, "cloud_cover": None}



In [ ]:
for idx, row in tqdm(photo_data_df.iterrows(), total=len(photo_data_df)):
    try:
        weather = get_weather(
            lat=row["lat"],
            lon=row["lon"],
            date_taken=row["datetime_taken"].strftime("%Y-%m-%d"),
            hour=row["hour"]
        )
        for key, value in weather.items():
            photo_data_df.at[idx, key] = value

    except Exception as e:
        print(f"Error at row {idx}: {e}")

photo_data_df


In [ ]:
csv_filepath = Path("data/csv/photo_data.csv") 
csv_filepath.parent.mkdir(parents=True, exist_ok=True)  
photo_data_df.to_csv(csv_filepath) 

In [ ]:
# create image from original image with photo exif data (from csv) displayed in lower left
    # create html page for image (jinja2 template?)
        # image (background)
        # data (white text or inverse of background?)
            # lat
            # lon
            # datetime
            # temp
        # other data: author name ? (in lower right)
    # save HTML image as a png or jpg in images/data_images


# Path to HTML Images Dir
html_images_dir = Path("html_images")
html_images_dir.mkdir(parents=True, exist_ok=True)

# Copy dataframe first to safely add new column data
full_photo_data_df = photo_data_df.copy()
full_photo_data_df['html_img_path'] = ""

# Loop Through Dataframe
for idx in full_photo_data_df.index:
    photo_path = full_photo_data_df["photo_path"][idx] 
    
    # Using pathlib to extract the name dynamically
    photo_name = Path(photo_path).stem 
    
    photo_lat = full_photo_data_df["lat"][idx]
    photo_lon = full_photo_data_df["lon"][idx]
    photo_date = full_photo_data_df["datetime_taken"][idx].strftime("%Y-%m-%d %H:%M")
    photo_temp = full_photo_data_df["temp_f"][idx]

    # HTML string
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>{photo_name}</title>
        <style>
            .body {{
                display: flex;
                flex-direction: column;
                align-items: center;
                background-color: black; 
            }}
            #image_div {{
                position: relative; /* anchors the absolute text inside */
                width: 80vw;
                height: auto;
                display: inline-block; /* Collapses container tightly around the image */
            }}
            img {{
                width: 100%;
                height: auto;
                display: block; /* Removes unwanted bottom whitespace under the image */
            }}
            #text_div {{
                position: absolute;
                bottom: 0;
                left: 0;
                box-sizing: border-box; 
                width: 50%; /* Spans the entire width of the photo for a clean edge-to-edge gradient */
                padding: 40px 20px 20px 20px; /* Extra top padding to cushion the fade-in */
                color: white;
                font-family: sans-serif;
                
                /* 45 degrees shifts the origin to the bottom-left corner */
                background: linear-gradient(45deg, rgba(0, 0, 0, 0.75) 0%, rgba(0, 0, 0, 0) 70%);
                
                /* Kept the shadow as a secondary fallback for ultra-bright backgrounds */
                text-shadow: 1px 1px 3px rgba(0, 0, 0, 0.5);
            }}
            #text_div p {{
                margin: 4px 0; /* Tightens the spacing between your metadata lines */
            }}
        </style>
    </head>
    <body class="body">
        <div id="image_div">
            <img src="../{photo_path}" alt="{photo_name}">
            <div id="text_div">
                <p>Lat: {photo_lat}</p>
                <p>Lon: {photo_lon}</p>
                <p>{photo_date}</p>
                <p>Temp: {photo_temp}°F</p>
            </div>
        </div>
    </body>
    </html>"""

    # Resolve the unique file path 
    html_filepath = f"{html_images_dir}/{photo_name}.html"

    # Save to HTML path
    with open(html_filepath, "w", encoding="utf-8") as file:
        file.write(html_content)

    # Save the individual path to the current row inside the loop
    full_photo_data_df.at[idx, 'html_img_path'] = str(html_filepath)

# Save the final CSV with all individual paths correctly set
full_photo_data_csv_filepath = Path("data/csv/full_photo_data.csv") 
full_photo_data_csv_filepath.parent.mkdir(parents=True, exist_ok=True)  
full_photo_data_df.to_csv(full_photo_data_csv_filepath, index=False) 


# Later:
    # save data_image_path to column with that name in photo_data.csv 

    # photo maps using geopandas -> maps/

In [ ]:
# loop at the beginning to replace image paths array